In [ ]:
import sys
sys.setrecursionlimit(30000)

from Bio import Phylo
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import re

CONSENSUS_TREE = "/data/users/ltucker/influenzaData/H5N1_pipeline/output/IQtree_consensus/consensus.contree"
MAPPING_CSV = "/data/users/ltucker/influenzaData/H5N1_pipeline/output/IQtree_consensus/tip_label_mapping.csv"
DR_DIR = "/data/users/ltucker/influenzaData/H5N1_pipeline/output/famsa_tensor_analysis/dimensionality_reduction"
FASTA_DIR = "/data/users/ltucker/influenzaData/H5N1_pipeline/output/segments"

CONTINENT_COLORS = {
    "africa": "#e6194b",
    "asia": "#3cb44b",
    "europe": "#4363d8",
    "americas": "#f58231",
    "oceania": "#911eb4",
}

In [ ]:
def extract_epi(header):
    m = re.search(r'EPI_ISL_\d+', str(header))
    return m.group(0) if m else None

# Load consensus tree
tree = Phylo.read(CONSENSUS_TREE, "newick")
print(f"Consensus tree: {tree.count_terminals()} tips")

# Load DR data for both factors
dr1 = pd.read_parquet(f"{DR_DIR}/dr_all_factor1_samples_row.parquet")
dr2 = pd.read_parquet(f"{DR_DIR}/dr_all_factor2_samples_col.parquet")

# Add epi_id to DR data
dr1["epi_id"] = dr1["sample_id"].apply(extract_epi)
dr2["epi_id"] = dr2["sample_id"].apply(extract_epi)

# Rename DR columns to distinguish factor1 vs factor2
dr1_renamed = dr1.rename(columns={
    "tsne_1": "f1_tsne_1", "tsne_2": "f1_tsne_2",
    "umap_1": "f1_umap_1", "umap_2": "f1_umap_2",
    "mds_1": "f1_mds_1",   "mds_2": "f1_mds_2",
})
dr2_renamed = dr2[["epi_id", "tsne_1", "tsne_2", "umap_1", "umap_2", "mds_1", "mds_2"]].rename(columns={
    "tsne_1": "f2_tsne_1", "tsne_2": "f2_tsne_2",
    "umap_1": "f2_umap_1", "umap_2": "f2_umap_2",
    "mds_1": "f2_mds_1",   "mds_2": "f2_mds_2",
})

# Merge factor1 metadata + embeddings with factor2 embeddings
df = dr1_renamed[["epi_id", "sample_id", "country", "continent", "year",
                   "f1_tsne_1", "f1_tsne_2", "f1_umap_1", "f1_umap_2",
                   "f1_mds_1", "f1_mds_2"]].copy()
df = df.merge(dr2_renamed, on="epi_id", how="left")

# Verify tree tips match
tree_epis = set(t.name for t in tree.get_terminals())
df_epis = set(df["epi_id"].dropna())
print(f"DR samples: {len(df)}")
print(f"In tree AND DR: {len(tree_epis & df_epis)}")
print(f"In tree only: {len(tree_epis - df_epis)}")
print(f"In DR only: {len(df_epis - tree_epis)}")

# Quick check
print(f"\nContinents: {sorted(df['continent'].dropna().unique())}")
print(f"Years: {int(df['year'].min())} – {int(df['year'].max())}")
df.head()

In [ ]:
def plot_by_year(tree, df):
    ymin, ymax = int(df["year"].min()), int(df["year"].max())
    norm = mcolors.Normalize(vmin=ymin, vmax=ymax)
    cmap = cm.viridis

    # Colour tree tips
    epi_to_year = dict(zip(df["epi_id"], df["year"]))
    for tip in tree.get_terminals():
        yr = epi_to_year.get(tip.name)
        if pd.notna(yr):
            tip.color = mcolors.to_hex(cmap(norm(int(yr))))
        else:
            tip.color = "#cccccc"
    for clade in tree.find_clades(terminal=False):
        clade.color = "#cccccc"

    # Remove bootstrap values and internal node labels
    for clade in tree.find_clades():
        clade.confidence = None
        if not clade.is_terminal():
            clade.name = None

    fig = plt.figure(figsize=(24, 20))
    gs = fig.add_gridspec(3, 3, height_ratios=[2, 1, 1], hspace=0.3)

    # Row 0: Tree
    ax_tree = fig.add_subplot(gs[0, :])
    Phylo.draw(tree, axes=ax_tree, do_show=False, label_func=lambda x: "")
    ax_tree.set_title("Consensus Phylogeny", fontsize=20)

    # Row 1: Factor 1
    f1_methods = [("f1_tsne_1", "f1_tsne_2", "Factor 1 — t-SNE"),
                  ("f1_umap_1", "f1_umap_2", "Factor 1 — UMAP"),
                  ("f1_mds_1",  "f1_mds_2",  "Factor 1 — MDS")]

    for col, (x, y, title) in enumerate(f1_methods):
        ax = fig.add_subplot(gs[1, col])
        valid = df.dropna(subset=[x, y, "year"])
        ax.scatter(valid[x], valid[y], c=valid["year"], cmap=cmap,
                   norm=norm, s=3, alpha=0.4)
        ax.set_title(title, fontsize=20)
        ax.set_xlabel(x, fontsize=15)
        ax.set_ylabel(y, fontsize=15)
        ax.tick_params(labelsize=10)

    # Row 2: Factor 2
    f2_methods = [("f2_tsne_1", "f2_tsne_2", "Factor 2 — t-SNE"),
                  ("f2_umap_1", "f2_umap_2", "Factor 2 — UMAP"),
                  ("f2_mds_1",  "f2_mds_2",  "Factor 2 — MDS")]

    for col, (x, y, title) in enumerate(f2_methods):
        ax = fig.add_subplot(gs[2, col])
        valid = df.dropna(subset=[x, y, "year"])
        ax.scatter(valid[x], valid[y], c=valid["year"], cmap=cmap,
                   norm=norm, s=3, alpha=0.4)
        ax.set_title(title, fontsize=20)
        ax.set_xlabel(x, fontsize=15)
        ax.set_ylabel(y, fontsize=15)
        ax.tick_params(labelsize=10)

    # Horizontal colorbar at bottom
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    cbar_ax = fig.add_axes([0.25, 0.02, 0.5, 0.015])
    cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
    cbar.set_label("Year", fontsize=20)

    plt.suptitle("Consensus Tree + Tucker Factor DR — by Year", fontsize=22)
    plt.subplots_adjust(bottom=0.07)
    plt.show()

plot_by_year(tree, df)

In [ ]:
def plot_by_continent(tree, df):
    epi_to_continent = dict(zip(df["epi_id"], df["continent"]))

    # Colour tree tips
    for tip in tree.get_terminals():
        continent = epi_to_continent.get(tip.name, "unknown")
        tip.color = CONTINENT_COLORS.get(continent, "#cccccc")
    for clade in tree.find_clades(terminal=False):
        clade.color = "#cccccc"

    # Remove bootstrap values and internal node labels
    for clade in tree.find_clades():
        clade.confidence = None
        if not clade.is_terminal():
            clade.name = None

    fig = plt.figure(figsize=(24, 20))
    gs = fig.add_gridspec(3, 3, height_ratios=[2, 1, 1], hspace=0.3)

    # Row 0: Tree spanning all columns
    ax_tree = fig.add_subplot(gs[0, :])
    Phylo.draw(tree, axes=ax_tree, do_show=False, label_func=lambda x: "")
    ax_tree.set_title("Consensus Phylogeny", fontsize=20)

    # Row 1: Factor 1 embeddings
    f1_methods = [("f1_tsne_1", "f1_tsne_2", "Factor 1 — t-SNE"),
                  ("f1_umap_1", "f1_umap_2", "Factor 1 — UMAP"),
                  ("f1_mds_1",  "f1_mds_2",  "Factor 1 — MDS")]

    for col, (x, y, title) in enumerate(f1_methods):
        ax = fig.add_subplot(gs[1, col])
        for continent in sorted(df["continent"].dropna().unique()):
            sub = df[df["continent"] == continent]
            color = CONTINENT_COLORS.get(continent, "#cccccc")
            ax.scatter(sub[x], sub[y], c=color, s=3, alpha=0.4, label=continent)
        ax.set_title(title, fontsize=20)
        ax.set_xlabel(x, fontsize=15)
        ax.set_ylabel(y, fontsize=15)
        ax.tick_params(labelsize=10)

    # Row 2: Factor 2 embeddings
    f2_methods = [("f2_tsne_1", "f2_tsne_2", "Factor 2 — t-SNE"),
                  ("f2_umap_1", "f2_umap_2", "Factor 2 — UMAP"),
                  ("f2_mds_1",  "f2_mds_2",  "Factor 2 — MDS")]

    for col, (x, y, title) in enumerate(f2_methods):
        ax = fig.add_subplot(gs[2, col])
        for continent in sorted(df["continent"].dropna().unique()):
            sub = df[df["continent"] == continent]
            color = CONTINENT_COLORS.get(continent, "#cccccc")
            ax.scatter(sub[x], sub[y], c=color, s=3, alpha=0.4, label=continent)
        ax.set_title(title, fontsize=20)
        ax.set_xlabel(x, fontsize=15)
        ax.set_ylabel(y, fontsize=15)
        ax.tick_params(labelsize=10)

    # Legend
    handles, labels = fig.axes[1].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=len(labels),
               fontsize=12, markerscale=4, framealpha=0.9)

    plt.suptitle("Consensus Tree + Tucker Factor DR — by Continent", fontsize=22)
    plt.subplots_adjust(bottom=0.05)
    plt.show()

plot_by_continent(tree, df)

## HDBSCAN and KMeans Clustering on Factor Matrices

In [ ]:
import re

CLUST_DIR = "/data/users/ltucker/influenzaData/H5N1_pipeline/output/famsa_tensor_analysis/clustering"

# HDBSCAN labels (raw factor = the one matching the full sample set)
hdb_f1 = pd.read_parquet(f"{CLUST_DIR}/hdbscan_factor1_raw.parquet")
hdb_f2 = pd.read_parquet(f"{CLUST_DIR}/hdbscan_factor2_raw.parquet")

# KMeans best-K by silhouette (raw factors)
km_f1 = pd.read_parquet(f"{CLUST_DIR}/kmeans_best_sil_factor1_raw_k9.parquet")
km_f2 = pd.read_parquet(f"{CLUST_DIR}/kmeans_best_sil_factor2_raw_k9.parquet")

# Add epi_id for merging
for frame in [hdb_f1, hdb_f2, km_f1, km_f2]:
    frame["epi_id"] = frame["sample_id"].apply(extract_epi)

# Merge into main df
df = df.copy()
df = df.merge(hdb_f1[["epi_id", "hdbscan_cluster"]].rename(columns={"hdbscan_cluster": "hdbscan_f1"}),
              on="epi_id", how="left")
df = df.merge(hdb_f2[["epi_id", "hdbscan_cluster"]].rename(columns={"hdbscan_cluster": "hdbscan_f2"}),
              on="epi_id", how="left")

# KMeans column name varies — find it
km_f1_col = [c for c in km_f1.columns if c.startswith("kmeans_k")][0]
km_f2_col = [c for c in km_f2.columns if c.startswith("kmeans_k")][0]
df = df.merge(km_f1[["epi_id", km_f1_col]].rename(columns={km_f1_col: "kmeans_f1"}),
              on="epi_id", how="left")
df = df.merge(km_f2[["epi_id", km_f2_col]].rename(columns={km_f2_col: "kmeans_f2"}),
              on="epi_id", how="left")

print(f"HDBSCAN F1: {df['hdbscan_f1'].nunique()} clusters (incl. noise=-1)")
print(f"HDBSCAN F2: {df['hdbscan_f2'].nunique()} clusters (incl. noise=-1)")
print(f"KMeans F1:  {df['kmeans_f1'].nunique()} clusters (K=9)")
print(f"KMeans F2:  {df['kmeans_f2'].nunique()} clusters (K=9)")
df.head()

In [ ]:
def make_cluster_colormap(labels):
    """Build a {cluster_id: hex_color} dict. Noise (-1) gets grey."""
    unique = sorted(set(labels.dropna().astype(int)))
    cmap = plt.get_cmap("tab20")
    color_map = {}
    real_clusters = [c for c in unique if c >= 0]
    for i, c in enumerate(real_clusters):
        color_map[c] = mcolors.to_hex(cmap(i % 20))
    color_map[-1] = "#cccccc"  # noise
    return color_map

In [ ]:
def plot_by_hdbscan(tree, df, factor="f1"):
    col = f"hdbscan_{factor}"
    labels = df[col].fillna(-1).astype(int)
    color_map = make_cluster_colormap(labels)
    n_clusters = len([c for c in color_map if c >= 0])

    # Colour tree tips
    epi_to_cluster = dict(zip(df["epi_id"], labels))
    for tip in tree.get_terminals():
        cl = epi_to_cluster.get(tip.name, -1)
        tip.color = color_map.get(cl, "#cccccc")
    for clade in tree.find_clades(terminal=False):
        clade.color = "#cccccc"
    for clade in tree.find_clades():
        clade.confidence = None
        if not clade.is_terminal():
            clade.name = None

    fig = plt.figure(figsize=(24, 20))
    gs = fig.add_gridspec(3, 3, height_ratios=[2, 1, 1], hspace=0.3)

    # Row 0: Tree
    ax_tree = fig.add_subplot(gs[0, :])
    Phylo.draw(tree, axes=ax_tree, do_show=False, label_func=lambda x: "")
    ax_tree.set_title(f"Consensus Phylogeny — HDBSCAN {factor.upper()}", fontsize=20)

    # Row 1: Factor 1 DR
    f1_methods = [("f1_tsne_1", "f1_tsne_2", "Factor 1 — t-SNE"),
                  ("f1_umap_1", "f1_umap_2", "Factor 1 — UMAP"),
                  ("f1_mds_1",  "f1_mds_2",  "Factor 1 — MDS")]
    for c, (x, y, title) in enumerate(f1_methods):
        ax = fig.add_subplot(gs[1, c])
        noise = df[labels == -1]
        ax.scatter(noise[x], noise[y], c="#cccccc", s=1, alpha=0.1, label="noise")
        for cl in sorted([k for k in color_map if k >= 0]):
            sub = df[labels == cl]
            ax.scatter(sub[x], sub[y], c=color_map[cl], s=3, alpha=0.4, label=f"C{cl}")
        ax.set_title(title, fontsize=20)
        ax.set_xlabel(x, fontsize=15)
        ax.set_ylabel(y, fontsize=15)
        ax.tick_params(labelsize=10)

    # Row 2: Factor 2 DR
    f2_methods = [("f2_tsne_1", "f2_tsne_2", "Factor 2 — t-SNE"),
                  ("f2_umap_1", "f2_umap_2", "Factor 2 — UMAP"),
                  ("f2_mds_1",  "f2_mds_2",  "Factor 2 — MDS")]
    for c, (x, y, title) in enumerate(f2_methods):
        ax = fig.add_subplot(gs[2, c])
        noise = df[labels == -1]
        ax.scatter(noise[x], noise[y], c="#cccccc", s=1, alpha=0.1, label="noise")
        for cl in sorted([k for k in color_map if k >= 0]):
            sub = df[labels == cl]
            ax.scatter(sub[x], sub[y], c=color_map[cl], s=3, alpha=0.4, label=f"C{cl}")
        ax.set_title(title, fontsize=20)
        ax.set_xlabel(x, fontsize=15)
        ax.set_ylabel(y, fontsize=15)
        ax.tick_params(labelsize=10)

    # Legend — only if manageable number of clusters
    if n_clusters <= 20:
        handles, lbls = fig.axes[1].get_legend_handles_labels()
        fig.legend(handles, lbls, loc="lower center", ncol=min(n_clusters + 1, 8),
                   fontsize=10, markerscale=3, framealpha=0.9)

    n_noise = (labels == -1).sum()
    plt.suptitle(f"HDBSCAN ({factor.upper()}) — {n_clusters} clusters, "
                 f"{n_noise} noise ({100*n_noise/len(labels):.1f}%)", fontsize=22)
    plt.subplots_adjust(bottom=0.06)
    plt.show()

plot_by_hdbscan(tree, df, "f1")
plot_by_hdbscan(tree, df, "f2")

In [ ]:
def plot_by_kmeans(tree, df, factor="f1"):
    col = f"kmeans_{factor}"
    labels = df[col].fillna(-1).astype(int)
    color_map = make_cluster_colormap(labels)
    n_clusters = len([c for c in color_map if c >= 0])

    # Colour tree tips
    epi_to_cluster = dict(zip(df["epi_id"], labels))
    for tip in tree.get_terminals():
        cl = epi_to_cluster.get(tip.name, -1)
        tip.color = color_map.get(cl, "#cccccc")
    for clade in tree.find_clades(terminal=False):
        clade.color = "#cccccc"
    for clade in tree.find_clades():
        clade.confidence = None
        if not clade.is_terminal():
            clade.name = None

    fig = plt.figure(figsize=(24, 20))
    gs = fig.add_gridspec(3, 3, height_ratios=[2, 1, 1], hspace=0.3)

    # Row 0: Tree
    ax_tree = fig.add_subplot(gs[0, :])
    Phylo.draw(tree, axes=ax_tree, do_show=False, label_func=lambda x: "")
    ax_tree.set_title(f"Consensus Phylogeny — KMeans {factor.upper()}", fontsize=20)

    # Row 1: Factor 1 DR
    f1_methods = [("f1_tsne_1", "f1_tsne_2", "Factor 1 — t-SNE"),
                  ("f1_umap_1", "f1_umap_2", "Factor 1 — UMAP"),
                  ("f1_mds_1",  "f1_mds_2",  "Factor 1 — MDS")]
    for c, (x, y, title) in enumerate(f1_methods):
        ax = fig.add_subplot(gs[1, c])
        for cl in sorted([k for k in color_map if k >= 0]):
            sub = df[labels == cl]
            ax.scatter(sub[x], sub[y], c=color_map[cl], s=3, alpha=0.4, label=f"C{cl}")
        ax.set_title(title, fontsize=20)
        ax.set_xlabel(x, fontsize=15)
        ax.set_ylabel(y, fontsize=15)
        ax.tick_params(labelsize=10)

    # Row 2: Factor 2 DR
    f2_methods = [("f2_tsne_1", "f2_tsne_2", "Factor 2 — t-SNE"),
                  ("f2_umap_1", "f2_umap_2", "Factor 2 — UMAP"),
                  ("f2_mds_1",  "f2_mds_2",  "Factor 2 — MDS")]
    for c, (x, y, title) in enumerate(f2_methods):
        ax = fig.add_subplot(gs[2, c])
        for cl in sorted([k for k in color_map if k >= 0]):
            sub = df[labels == cl]
            ax.scatter(sub[x], sub[y], c=color_map[cl], s=3, alpha=0.4, label=f"C{cl}")
        ax.set_title(title, fontsize=20)
        ax.set_xlabel(x, fontsize=15)
        ax.set_ylabel(y, fontsize=15)
        ax.tick_params(labelsize=10)

    # Legend
    if n_clusters <= 20:
        handles, lbls = fig.axes[1].get_legend_handles_labels()
        fig.legend(handles, lbls, loc="lower center", ncol=min(n_clusters, 8),
                   fontsize=10, markerscale=3, framealpha=0.9)

    plt.suptitle(f"KMeans ({factor.upper()}, K={n_clusters})", fontsize=22)
    plt.subplots_adjust(bottom=0.06)
    plt.show()

plot_by_kmeans(tree, df, "f1")
plot_by_kmeans(tree, df, "f2")

## HDBSCAN and KMeans Clustering on Embeddings

In [ ]:
CLUST_DIR = "/data/users/ltucker/influenzaData/H5N1_pipeline/output/famsa_tensor_analysis/clustering"

# HDBSCAN on 2D embeddings
hdb_f1_tsne = pd.read_parquet(f"{CLUST_DIR}/hdbscan_factor1_tsne_2d.parquet")
hdb_f1_umap = pd.read_parquet(f"{CLUST_DIR}/hdbscan_factor1_umap_2d.parquet")
hdb_f1_mds  = pd.read_parquet(f"{CLUST_DIR}/hdbscan_factor1_mds_2d.parquet")
hdb_f2_tsne = pd.read_parquet(f"{CLUST_DIR}/hdbscan_factor2_tsne_2d.parquet")
hdb_f2_umap = pd.read_parquet(f"{CLUST_DIR}/hdbscan_factor2_umap_2d.parquet")
hdb_f2_mds  = pd.read_parquet(f"{CLUST_DIR}/hdbscan_factor2_mds_2d.parquet")

# KMeans on 2D embeddings (best K by silhouette)
km_f1_tsne = pd.read_parquet(f"{CLUST_DIR}/kmeans_best_sil_factor1_tsne_2d_k13.parquet")
km_f1_umap = pd.read_parquet(f"{CLUST_DIR}/kmeans_best_sil_factor1_umap_2d_k9.parquet")
km_f1_mds  = pd.read_parquet(f"{CLUST_DIR}/kmeans_best_sil_factor1_mds_2d_k14.parquet")
km_f2_tsne = pd.read_parquet(f"{CLUST_DIR}/kmeans_best_sil_factor2_tsne_2d_k13.parquet")
km_f2_umap = pd.read_parquet(f"{CLUST_DIR}/kmeans_best_sil_factor2_umap_2d_k7.parquet")
km_f2_mds  = pd.read_parquet(f"{CLUST_DIR}/kmeans_best_sil_factor2_mds_2d_k12.parquet")

# Add epi_id to all
all_frames = [hdb_f1_tsne, hdb_f1_umap, hdb_f1_mds, hdb_f2_tsne, hdb_f2_umap, hdb_f2_mds,
              km_f1_tsne, km_f1_umap, km_f1_mds, km_f2_tsne, km_f2_umap, km_f2_mds]
for frame in all_frames:
    frame["epi_id"] = frame["sample_id"].apply(extract_epi)

# Merge all into df — each DR method gets its own cluster column
df_2d = df[["epi_id", "sample_id", "country", "continent", "year",
            "f1_tsne_1", "f1_tsne_2", "f1_umap_1", "f1_umap_2", "f1_mds_1", "f1_mds_2",
            "f2_tsne_1", "f2_tsne_2", "f2_umap_1", "f2_umap_2", "f2_mds_1", "f2_mds_2"]].copy()

# HDBSCAN
df_2d = df_2d.merge(hdb_f1_tsne[["epi_id", "hdbscan_cluster"]].rename(columns={"hdbscan_cluster": "hdb_f1_tsne"}), on="epi_id", how="left")
df_2d = df_2d.merge(hdb_f1_umap[["epi_id", "hdbscan_cluster"]].rename(columns={"hdbscan_cluster": "hdb_f1_umap"}), on="epi_id", how="left")
df_2d = df_2d.merge(hdb_f1_mds[["epi_id", "hdbscan_cluster"]].rename(columns={"hdbscan_cluster": "hdb_f1_mds"}), on="epi_id", how="left")
df_2d = df_2d.merge(hdb_f2_tsne[["epi_id", "hdbscan_cluster"]].rename(columns={"hdbscan_cluster": "hdb_f2_tsne"}), on="epi_id", how="left")
df_2d = df_2d.merge(hdb_f2_umap[["epi_id", "hdbscan_cluster"]].rename(columns={"hdbscan_cluster": "hdb_f2_umap"}), on="epi_id", how="left")
df_2d = df_2d.merge(hdb_f2_mds[["epi_id", "hdbscan_cluster"]].rename(columns={"hdbscan_cluster": "hdb_f2_mds"}), on="epi_id", how="left")

# KMeans — find the cluster column name dynamically
for frame, col_name in [(km_f1_tsne, "km_f1_tsne"), (km_f1_umap, "km_f1_umap"), (km_f1_mds, "km_f1_mds"),
                         (km_f2_tsne, "km_f2_tsne"), (km_f2_umap, "km_f2_umap"), (km_f2_mds, "km_f2_mds")]:
    km_col = [c for c in frame.columns if c.startswith("kmeans_k")][0]
    df_2d = df_2d.merge(frame[["epi_id", km_col]].rename(columns={km_col: col_name}), on="epi_id", how="left")

print(f"Shape: {df_2d.shape}")
print("\nHDBSCAN cluster counts:")
for c in ["hdb_f1_tsne", "hdb_f1_umap", "hdb_f1_mds", "hdb_f2_tsne", "hdb_f2_umap", "hdb_f2_mds"]:
    n = df_2d[c].nunique()
    noise = (df_2d[c] == -1).sum()
    print(f"  {c}: {n} unique ({noise} noise)")
print("\nKMeans cluster counts:")
for c in ["km_f1_tsne", "km_f1_umap", "km_f1_mds", "km_f2_tsne", "km_f2_umap", "km_f2_mds"]:
    print(f"  {c}: K={int(df_2d[c].nunique())}")

In [ ]:
def plot_hdbscan_2d(tree, df_2d):
    """Tree on top, 2 rows of DR scatter plots, each coloured by its own HDBSCAN clustering."""

    # Colour tree by HDBSCAN on factor1 t-SNE (arbitrary choice for tree)
    tree_col = "hdb_f1_tsne"
    labels = df_2d[tree_col].fillna(-1).astype(int)
    tree_cmap = make_cluster_colormap(labels)
    epi_to_cl = dict(zip(df_2d["epi_id"], labels))
    for tip in tree.get_terminals():
        cl = epi_to_cl.get(tip.name, -1)
        tip.color = tree_cmap.get(cl, "#cccccc")
    for clade in tree.find_clades(terminal=False):
        clade.color = "#cccccc"
    for clade in tree.find_clades():
        clade.confidence = None
        if not clade.is_terminal():
            clade.name = None

    fig = plt.figure(figsize=(24, 20))
    gs = fig.add_gridspec(3, 3, height_ratios=[2, 1, 1], hspace=0.3)

    # Row 0: Tree
    ax_tree = fig.add_subplot(gs[0, :])
    Phylo.draw(tree, axes=ax_tree, do_show=False, label_func=lambda x: "")
    ax_tree.set_title(f"Consensus Phylogeny — coloured by HDBSCAN (F1 t-SNE)", fontsize=20)

    # Row 1: Factor 1 — each panel coloured by its own HDBSCAN
    f1_configs = [
        ("f1_tsne_1", "f1_tsne_2", "hdb_f1_tsne", "F1 t-SNE (HDBSCAN)"),
        ("f1_umap_1", "f1_umap_2", "hdb_f1_umap", "F1 UMAP (HDBSCAN)"),
        ("f1_mds_1",  "f1_mds_2",  "hdb_f1_mds",  "F1 MDS (HDBSCAN)"),
    ]
    for c, (x, y, cl_col, title) in enumerate(f1_configs):
        ax = fig.add_subplot(gs[1, c])
        lab = df_2d[cl_col].fillna(-1).astype(int)
        cmap_local = make_cluster_colormap(lab)
        noise = df_2d[lab == -1]
        ax.scatter(noise[x], noise[y], c="#cccccc", s=1, alpha=0.1)
        for cl in sorted([k for k in cmap_local if k >= 0]):
            sub = df_2d[lab == cl]
            ax.scatter(sub[x], sub[y], c=cmap_local[cl], s=3, alpha=0.4, label=f"C{cl}")
        n_cl = len([k for k in cmap_local if k >= 0])
        ax.set_title(f"{title} — {n_cl} clusters", fontsize=16)
        ax.set_xlabel(x, fontsize=15)
        ax.set_ylabel(y, fontsize=15)
        ax.tick_params(labelsize=10)

    # Row 2: Factor 2
    f2_configs = [
        ("f2_tsne_1", "f2_tsne_2", "hdb_f2_tsne", "F2 t-SNE (HDBSCAN)"),
        ("f2_umap_1", "f2_umap_2", "hdb_f2_umap", "F2 UMAP (HDBSCAN)"),
        ("f2_mds_1",  "f2_mds_2",  "hdb_f2_mds",  "F2 MDS (HDBSCAN)"),
    ]
    for c, (x, y, cl_col, title) in enumerate(f2_configs):
        ax = fig.add_subplot(gs[2, c])
        lab = df_2d[cl_col].fillna(-1).astype(int)
        cmap_local = make_cluster_colormap(lab)
        noise = df_2d[lab == -1]
        ax.scatter(noise[x], noise[y], c="#cccccc", s=1, alpha=0.1)
        for cl in sorted([k for k in cmap_local if k >= 0]):
            sub = df_2d[lab == cl]
            ax.scatter(sub[x], sub[y], c=cmap_local[cl], s=3, alpha=0.4, label=f"C{cl}")
        n_cl = len([k for k in cmap_local if k >= 0])
        ax.set_title(f"{title} — {n_cl} clusters", fontsize=16)
        ax.set_xlabel(x, fontsize=15)
        ax.set_ylabel(y, fontsize=15)
        ax.tick_params(labelsize=10)

    plt.suptitle("HDBSCAN on 2D Embeddings — each panel coloured by its own clustering", fontsize=22)
    plt.subplots_adjust(bottom=0.03)
    plt.show()

plot_hdbscan_2d(tree, df_2d)

In [ ]:
def plot_kmeans_2d(tree, df_2d):
    """Tree on top, 2 rows of DR scatter plots, each coloured by its own KMeans clustering."""

    # Colour tree by KMeans on factor1 t-SNE
    tree_col = "km_f1_tsne"
    labels = df_2d[tree_col].fillna(-1).astype(int)
    tree_cmap = make_cluster_colormap(labels)
    epi_to_cl = dict(zip(df_2d["epi_id"], labels))
    for tip in tree.get_terminals():
        cl = epi_to_cl.get(tip.name, -1)
        tip.color = tree_cmap.get(cl, "#cccccc")
    for clade in tree.find_clades(terminal=False):
        clade.color = "#cccccc"
    for clade in tree.find_clades():
        clade.confidence = None
        if not clade.is_terminal():
            clade.name = None

    fig = plt.figure(figsize=(24, 20))
    gs = fig.add_gridspec(3, 3, height_ratios=[2, 1, 1], hspace=0.3)

    # Row 0: Tree
    ax_tree = fig.add_subplot(gs[0, :])
    Phylo.draw(tree, axes=ax_tree, do_show=False, label_func=lambda x: "")
    ax_tree.set_title(f"Consensus Phylogeny — coloured by KMeans (F1 t-SNE)", fontsize=20)

    # Row 1: Factor 1
    f1_configs = [
        ("f1_tsne_1", "f1_tsne_2", "km_f1_tsne", "F1 t-SNE"),
        ("f1_umap_1", "f1_umap_2", "km_f1_umap", "F1 UMAP"),
        ("f1_mds_1",  "f1_mds_2",  "km_f1_mds",  "F1 MDS"),
    ]
    for c, (x, y, cl_col, title) in enumerate(f1_configs):
        ax = fig.add_subplot(gs[1, c])
        lab = df_2d[cl_col].fillna(-1).astype(int)
        cmap_local = make_cluster_colormap(lab)
        k = len([k for k in cmap_local if k >= 0])
        for cl in sorted([k for k in cmap_local if k >= 0]):
            sub = df_2d[lab == cl]
            ax.scatter(sub[x], sub[y], c=cmap_local[cl], s=3, alpha=0.4, label=f"C{cl}")
        ax.set_title(f"{title} (KMeans K={k})", fontsize=16)
        ax.set_xlabel(x, fontsize=15)
        ax.set_ylabel(y, fontsize=15)
        ax.tick_params(labelsize=10)

    # Row 2: Factor 2
    f2_configs = [
        ("f2_tsne_1", "f2_tsne_2", "km_f2_tsne", "F2 t-SNE"),
        ("f2_umap_1", "f2_umap_2", "km_f2_umap", "F2 UMAP"),
        ("f2_mds_1",  "f2_mds_2",  "km_f2_mds",  "F2 MDS"),
    ]
    for c, (x, y, cl_col, title) in enumerate(f2_configs):
        ax = fig.add_subplot(gs[2, c])
        lab = df_2d[cl_col].fillna(-1).astype(int)
        cmap_local = make_cluster_colormap(lab)
        k = len([k for k in cmap_local if k >= 0])
        for cl in sorted([k for k in cmap_local if k >= 0]):
            sub = df_2d[lab == cl]
            ax.scatter(sub[x], sub[y], c=cmap_local[cl], s=3, alpha=0.4, label=f"C{cl}")
        ax.set_title(f"{title} (KMeans K={k})", fontsize=16)
        ax.set_xlabel(x, fontsize=15)
        ax.set_ylabel(y, fontsize=15)
        ax.tick_params(labelsize=10)

    plt.suptitle("KMeans on 2D Embeddings — each panel coloured by its own clustering", fontsize=22)
    plt.subplots_adjust(bottom=0.03)
    plt.show()

plot_kmeans_2d(tree, df_2d)